# The human gene pack


**What this builds (instruction 64.21).** A data pack for the human gene space: the gene table (58,988 GTEx v10 genes with HPA annotations), a 3-D layout of the genes, the reviewed gene-protein links, the description of every column and the attribution notice. Each column carries its licence: GTEx and UniProt-derived columns are open with attribution (O); HPA columns are share-alike (SA).

**Then it is opened on its own:** installed into an empty cache and read back through the same calls ParaScape uses for any pack space.

Built 2026-11-07.

In [1]:
import json, os, shutil, numpy as np, pandas as pd
from parascape import organisms as O, packs, paths
from parascape.spaces import build_pack, validate_graph, validate_nodes
from parascape.spaces.human import HUMAN_SPACE
nodes = pd.read_parquet(os.path.join(SPACE_DIR, 'hs_genes.parquet'))
meta = json.load(open(os.path.join(SPACE_DIR, 'column_metadata.json')))
validate_nodes(HUMAN_SPACE, nodes)
print(len(nodes), 'genes,', nodes.shape[1], 'columns')

58988 genes, 12 columns


## 1 · Where each gene sits

Three coordinates per gene from log expression in the two GTEx contexts and the HPA categories (tissue specificity and distribution, single-cell specificity, subcellular location, secretome location, protein class), one indicator per category value. UMAP, cosine distance, 15 neighbours, fixed seed. Genes without any HPA annotation are placed by expression alone. The layout is for display; it is not an input to any answer.

In [2]:
import umap
cats = [c for c in nodes.columns if c.startswith('HPA_')]
parts = [np.log1p(nodes[['gtex_fibroblast_median_tpm', 'gtex_lcm_hepatocyte_median_tpm']].fillna(0)).to_numpy()]
for c in cats:
    tokens = nodes[c].fillna('').astype(str).str.replace(';', ',').str.split(',')
    parts.append(pd.get_dummies(tokens.explode().str.strip()).groupby(level=0).max().drop(columns='', errors='ignore').to_numpy(float))
X = np.hstack(parts)
X = X + 1e-6 * np.random.default_rng(0).standard_normal(X.shape)  # no all-zero rows under cosine distance
xyz = umap.UMAP(n_components=3, n_neighbors=15, metric='cosine', random_state=0, low_memory=True).fit_transform(X).astype(np.float32)
build = os.path.join(OUT, 'build'); os.makedirs(build, exist_ok=True)
nodes.to_parquet(os.path.join(build, HUMAN_SPACE.nodes), index=False)
np.savez(os.path.join(build, HUMAN_SPACE.graph), xyz=xyz, gene_ids=nodes.gene_id.to_numpy(dtype=str))
validate_graph(nodes, os.path.join(build, HUMAN_SPACE.graph))
print(X.shape[1], 'features; layout', xyz.shape)

106 features; layout (58988, 3)


## 2 · Licence per column and the attribution notice

From the documentary partition (`results/human_reference_terms_2026_10_08_v5/`): GTEx public data may be redistributed with source and date acknowledged, no implied endorsement, and a notice when the version is not current; UniProt is CC BY 4.0; the Human Protein Atlas is CC BY-SA 4.0.

In [3]:
lic = {c: ('SA' if c.startswith('HPA_') else 'O') for c in nodes.columns}
notice = '\n'.join([
    f'ParaScape human gene pack {VERSION}',
    '',
    'Genes and expression: GTEx Analysis Release V10 (dbGaP phs000424), median TPM, GENCODE 39 / GRCh38.p13.',
    '  The Genotype-Tissue Expression (GTEx) Project was supported by the Common Fund of the Office of the Director of the NIH.',
    '  Data retrieved 2026-10-08 from https://gtexportal.org. Use does not imply endorsement by GTEx or the NIH.',
    '  This pack pins GTEx V10; newer GTEx releases may exist (check https://gtexportal.org/home/releaseInfoPage).',
    'Gene-protein links: UniProt Knowledgebase release 2026_03, reviewed human entries, CC BY 4.0 (https://www.uniprot.org).',
    'Annotations (HPA_ columns): Human Protein Atlas (https://www.proteinatlas.org), CC BY-SA 4.0;',
    '  those columns, and works adapted from them, must be shared under the same licence.',
    '',
    'Licence per column: ' + ', '.join(f'{c}={v}' for c, v in lic.items()),
]) + '\n'
open(os.path.join(build, 'NOTICE.txt'), 'w').write(notice)
print(notice)

ParaScape human gene pack 2026.11.07

Genes and expression: GTEx Analysis Release V10 (dbGaP phs000424), median TPM, GENCODE 39 / GRCh38.p13.
  The Genotype-Tissue Expression (GTEx) Project was supported by the Common Fund of the Office of the Director of the NIH.
  Data retrieved 2026-10-08 from https://gtexportal.org. Use does not imply endorsement by GTEx or the NIH.
  This pack pins GTEx V10; newer GTEx releases may exist (check https://gtexportal.org/home/releaseInfoPage).
Gene-protein links: UniProt Knowledgebase release 2026_03, reviewed human entries, CC BY 4.0 (https://www.uniprot.org).
Annotations (HPA_ columns): Human Protein Atlas (https://www.proteinatlas.org), CC BY-SA 4.0;
  those columns, and works adapted from them, must be shared under the same licence.

Licence per column: gene_id=O, gene_name=O, gtex_fibroblast_median_tpm=O, gtex_lcm_hepatocyte_median_tpm=O, HPA_RNA_tissue_specificity=SA, HPA_RNA_tissue_distribution=SA, HPA_RNA_singlecell_specificity=SA, HPA_subcell

## 3 · The pack

In [4]:
extra = {'NOTICE.txt': os.path.join(build, 'NOTICE.txt'),
         'column_metadata.json': os.path.join(SPACE_DIR, 'column_metadata.json'),
         'hs_protein_gene_links.parquet': os.path.join(SPACE_DIR, 'hs_protein_gene_links.parquet')}
archive = os.path.join(OUT, f'parascape_Hs_{VERSION}.zip')
sha = build_pack(HUMAN_SPACE, build, VERSION, lic, archive, extra=extra)
manifest = packs.verify(archive, sha)
open(os.path.join(OUT, 'SHA256'), 'w').write(f'{sha}  {os.path.basename(archive)}\n')
print(os.path.getsize(archive) // 1024, 'KiB;', sha)
sorted(manifest['files'])

2505 KiB; 21f7d87f4b39de3b420801bda9db9e3d9ab59e25f466b30e202fd70c08fd4462


['NOTICE.txt', 'column_metadata.json', 'hs_gene_graph.npz', 'hs_gene_nodes.parquet', 'hs_protein_gene_links.parquet']

## 4 · Opening it on its own

An empty cache, the pack installed with its checksum, and the gene table and layout read back through `organisms.nodes_path` / `graph_path` -- the calls the application uses.

In [5]:
shutil.rmtree(STATE, ignore_errors=True)
os.environ[paths.ENV_STATE] = STATE
added = 'Hs' not in O.SPACES
if added: O.SPACES['Hs'] = HUMAN_SPACE
try:
    where = packs.install(archive, sha, 'Hs')
    back = pd.read_parquet(O.nodes_path('Hs'))
    validate_graph(back, O.graph_path('Hs'))
    opened = dict(installed_at=os.path.relpath(str(where), STATE), genes=len(back),
                  identical=bool(back.equals(nodes)), notice=os.path.exists(where / 'NOTICE.txt'),
                  protein_rows_as_genes=int(back.gene_id.isin(pd.read_parquet(where / 'hs_protein_gene_links.parquet').host_id).sum()))
finally:
    if added: O.SPACES.pop('Hs')
    os.environ.pop(paths.ENV_STATE)
    shutil.rmtree(STATE, ignore_errors=True)
assert opened['identical'] and opened['notice'] and opened['protein_rows_as_genes'] == 0
json.dump(dict(version=VERSION, sha256=sha, archive=os.path.basename(archive), licences=lic, opened=opened),
          open(os.path.join(OUT, 'pack_report.json'), 'w'), indent=1)
shutil.rmtree(build)
opened

{'installed_at': 'spaces/Hs/2026.11.07', 'genes': 58988, 'identical': True, 'notice': True, 'protein_rows_as_genes': 0}